# Lab 6: Prompt Variants & A/B Testing

**Duration:** 30 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Create multiple system prompt variants for the banking RAG
- Run the same queries against different prompts
- Compare response quality, length, and citation behavior
- Select the best prompt variant based on metrics

## Prerequisites

Labs 1-3 completed

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Step 2: Define Helper Functions

In [ ]:
def call_openai(messages, max_tokens=150, temperature=0.1):
    url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = json.dumps({'messages':messages,'max_tokens':max_tokens,'temperature':temperature}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        result = json.loads(resp.read())
    return result['choices'][0]['message']['content'], result.get('usage', {})

def get_embedding(text):
    url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = json.dumps({'input':text}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['data'][0]['embedding']

def hybrid_search(query, top_k=3):
    vector = get_embedding(query)
    url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
    body = {'search':query,'vectorQueries':[{'vector':vector,'k':top_k,'fields':'content_vector','kind':'vector'}],
            'select':'title,content,category,source_file','top':top_k}
    data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method='POST',
        headers={'Content-Type':'application/json','api-key':SEARCH_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read()).get('value', [])

print('✅ Helper functions defined.')

## Step 3: Define Prompt Variants

Four variants with different styles:
- **A_concise** — Short 2-3 sentence answers
- **B_detailed** — Comprehensive with caveats
- **C_structured** — Formatted with Answer/Details/Source/Note
- **D_compliance** — Includes disclaimers and exact policy terms

In [ ]:
VARIANTS = {
    'A_concise': 'You are a SecureBank assistant. Answer using ONLY the provided context. Be concise (2-3 sentences max). Cite sources as [Source: filename]. If unsure, say so.',

    'B_detailed': 'You are a SecureBank banking policy expert. Provide comprehensive answers using the provided context. Give detailed explanations with specific numbers. Always cite sources using [Source: filename]. Include relevant caveats. If context lacks the answer, state that clearly.',

    'C_structured': '''You are a SecureBank assistant. Answer using ONLY the provided context.
Format: **Answer:** [1-2 sentences] **Details:** [specifics] **Source:** [document] **Note:** [caveats]
If context lacks the answer, say "Information not available."''',

    'D_compliance': 'You are a SecureBank compliance-aware assistant. Answer using ONLY the provided context. Include exact policy terms and conditions. Cite specific document and section. Add compliance disclaimer for financial products. Never provide personal financial advice.',
}

test_questions = [
    'What is the wire transfer limit for business accounts?',
    'How do I dispute a transaction?',
    'What savings account has the best interest rate?',
    'What should I do if I suspect fraud?',
]

print(f'Variants: {len(VARIANTS)}')
print(f'Test questions: {len(test_questions)}')

## Step 4: Run A/B Test

In [ ]:
print('=' * 70)
print('  PROMPT VARIANT A/B TEST — Banking RAG')
print('=' * 70)

all_results = {}
for variant_name, system_prompt in VARIANTS.items():
    print(f'\n{"─"*70}')
    print(f'  VARIANT {variant_name}')
    print(f'{"─"*70}')
    variant_results = []
    total_tokens = 0
    for q in test_questions:
        results = hybrid_search(q)
        context = '\n\n'.join([f'[{r["source_file"]}] {r["title"]}: {r["content"]}' for r in results])
        start = time.time()
        answer, usage = call_openai([{'role':'system','content':system_prompt},
            {'role':'user','content':f'Context:\n{context}\n\nQuestion: {q}'}])
        latency = time.time() - start
        tokens = usage.get('total_tokens', 0)
        total_tokens += tokens
        has_cite = '[Source:' in answer or any(r['source_file'] in answer for r in results)
        wc = len(answer.split())
        variant_results.append({'question':q,'answer':answer,'tokens':tokens,
            'latency':round(latency,2),'word_count':wc,'has_citation':has_cite})
        print(f'\n  Q: {q}')
        print(f'  A: {answer[:150]}...')
        print(f'  📊 Words:{wc} | Tokens:{tokens} | Latency:{latency:.1f}s | Citation:{"✅" if has_cite else "❌"}')
        time.sleep(0.3)
    n = len(variant_results)
    all_results[variant_name] = {
        'total_tokens':total_tokens,
        'avg_words':round(sum(r['word_count'] for r in variant_results)/n, 1),
        'avg_latency':round(sum(r['latency'] for r in variant_results)/n, 2),
        'citation_rate':round(sum(1 for r in variant_results if r['has_citation'])/n*100, 1)
    }

## Step 5: Comparison Summary

In [ ]:
print('=' * 70)
print('  VARIANT COMPARISON SUMMARY')
print('=' * 70)
print(f'{"Variant":<18} {"Avg Words":<12} {"Avg Latency":<14} {"Tokens":<10} {"Citations":<10}')
print('-' * 64)
for name, data in all_results.items():
    print(f'{name:<18} {data["avg_words"]:<12} {data["avg_latency"]:<14}s {data["total_tokens"]:<10} {data["citation_rate"]}%')

best = max(all_results.items(), key=lambda x: x[1]['citation_rate'])
print(f'\n🏆 Recommended: {best[0]} (citation rate: {best[1]["citation_rate"]}%)')

print(f'\n💰 Cost Estimate (per 10,000 queries):')
for name, data in all_results.items():
    avg_tok = data['total_tokens'] / len(test_questions)
    cost = avg_tok * 10000 / 1000 * 0.005
    print(f'   {name}: ~${cost:.2f}')

## ✅ Lab 6 Complete

**What you accomplished:**
- Created 4 prompt variants (concise, detailed, structured, compliance)
- Ran A/B test across 4 banking questions
- Compared word count, latency, token usage, citation rates
- Generated cost estimates per variant

**Next:** Open `promptflow_lab7_deploy_endpoint.ipynb`


> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.
---

© 2026 Great Learning. All rights reserved.